# 19th-Century Almshouses (Frechet Spatial Modeling)
In the 19th and early 20th centuries, there was a major shift from "outdoor relief" (giving food or money to people in their own homes) to "indoor relief" (requiring people to live in an institution to receive help). The explicit goal was often to make the experience of being "on the town" so unpleasant--through hard labor like picking oakum or stone-breaking--that only the truly desperate would apply, thereby discouraging "pauperism."

This notebook adapts the **Quantitative Spatial Economics (QSE)** Frechet framework to model this historical dynamic. 

## 1. The $2N$ Choice Set Model: Verbal Description

In a standard spatial model, a person born in county $i$ simply chooses which of the $N$ counties they want to live and work in. To model 19th-century homelessness, we expand their choices: an individual now faces **$2N$ options**. 

For every county $j$, they must choose both a location and a status:
1.  **Employment:** The individual enters the regular labor market in county $j$. Their baseline well-being is driven by the local wage $w_j$.
2.  **Almshouse:** The individual becomes destitute and enters the indoor relief system in county $j$. Their baseline well-being is driven by the net relief benefit $B_j$, which represents the value of food and shelter *minus* the harshness and social stigma of the institution.

*(Note: We omit explicit location "amenities" $A_j$ here for parsimony, but they can easily be multiplied into the utility functions later to capture unobserved local quality of life).*

Why would anyone choose the almshouse if the relief benefit $B_j$ is fundamentally worse than the wage $w_j$? Because humans experience idiosyncratic life shocks. In this model, people choose the option that maximizes their *total* utility, which is the baseline economic payoff multiplied by a random personal life shock. If an individual suffers a catastrophic life event (like a crippling factory injury or severe illness), their personal "shock draw" for employment approaches zero, destroying their ability to earn the wage. In that extreme scenario, their total utility for employment collapses, and the harsh almshouse becomes their only surviving option.



## 2. Mathematical Formalization

### Step 1: Production and Deterministic Values
In each county $j$, there is a fixed amount of capital/land $K_j$. Due to competitive labor markets and a Cobb-Douglas production function, the wage $w_j$ equals the marginal product of labor:
$$ w_j = (1 - \alpha) K_j^\alpha L_j^{-\alpha} $$
*(Notice that as the employed population $L_j$ increases, the wage drops due to diminishing returns).*

An individual born in county $i$ faces a migration friction $\mu_{ij} = \exp(\kappa \cdot d_{ij})$ to travel to county $j$. The deterministic (baseline) economic value of their choices is simply the local payoff deflated by this travel friction:
*   **Value of Employment (Status E):** $V_{ijE} = w_j / \mu_{ij}$
*   **Value of Almshouse (Status H):** $V_{ijH} = B_j / \mu_{ij}$

### Step 2: Individual Utility and the Frechet Draw
An individual $k$ born in county $i$ draws an independent, random life shock $\epsilon$ for every possible location $j$ and status $s \in \{E, H\}$. These shocks are drawn from a Frechet distribution with the cumulative density function:
$$ F(\epsilon) = \exp(-\epsilon^{-\theta}) $$

The individual calculates their total utility for each option by multiplying the deterministic value by their personal shock:
*   $U_{k,j,E} = V_{ijE} \times \epsilon_{k,j,E}$
*   $U_{k,j,H} = V_{ijH} \times \epsilon_{k,j,H}$

The individual then selects the single option that provides the absolute highest total utility $U$. Because the relief benefit $B_j$ is so low, $V_{ijH}$ is much smaller than $V_{ijE}$. Therefore, the Almshouse ($U_{k,j,H}$) almost never wins for a healthy person. It only dominates if the individual draws a devastatingly low $\epsilon_{k,j,E}$ (e.g., an injury that crushes their ability to earn a wage), causing their employment utility $U_{k,j,E}$ to collapse.

### Step 3: Aggregate Probabilities (The "Frechet Trick")
Because the $\epsilon$ shocks are Frechet-distributed, the complex math of aggregating everyone's "maximum" choice simplifies perfectly. The aggregate probability $\pi$ that a person born in county $i$ chooses a specific option is simply the ratio of that option's deterministic value (raised to the power of $\theta$) to the sum of *all* $2N$ available options:

$$ \pi_{ijE} = \frac{ (w_j / \mu_{ij})^\theta }{ \sum_{k=1}^N (w_k / \mu_{ik})^\theta + \sum_{k=1}^N (B_k / \mu_{ik})^\theta } $$

$$ \pi_{ijH} = \frac{ (B_j / \mu_{ij})^\theta }{ \sum_{k=1}^N (w_k / \mu_{ik})^\theta + \sum_{k=1}^N (B_k / \mu_{ik})^\theta } $$

### Step 4: Market Clearing
To find the total employed population ($L_j$) and total almshouse population ($H_j$) in a given county, we sum up all the migrants arriving from every origin $i$ (including the "stayers" who never left $i=j$):
$$ L_j = \sum_{i=1}^N \pi_{ijE} L_{0,i} $$
$$ H_j = \sum_{i=1}^N \pi_{ijH} L_{0,i} $$
where $L_{0,i}$ is the initial population born in county $i$. Because $w_j$ depends on $L_j$, and $L_j$ depends on $w_j$, this forms a system of equations that we solve computationally using a dampened fixed-point iteration loop.



## 3. The Engine of the Model: The Frechet Distribution

To aggregate individual choices into a macroeconomic model, we assumed the idiosyncratic life shocks ($\epsilon$) were drawn from a **Frechet distribution**. This statistical distribution provides several powerful properties for modeling 19th-century pauperism:

1. **Extreme Values (Life Shocks):** The Frechet is mathematically designed to model the "maximums" of random variables. It natively captures catastrophic tail-events--like a crippling factory injury or sudden disease--that would force an otherwise rational person to abandon the labor market and enter a harsh almshouse out of sheer desperation. 
2. **Max-Stability (The "Frechet Trick"):** An individual simultaneously evaluates all $2N$ independent options across the map and chooses the single option with the absolute maximum utility. The most beautiful property of the Frechet distribution is *max-stability*: the maximum of independent Frechet variables is *itself* a Frechet variable! This mathematical "trick" avoids intractable integrals; the math elegantly collapses, transforming thousands of chaotic, individual life shocks into the smooth, continuous choice probabilities ($\pi$) shown above.
3. **Variance and Elasticity ($\theta$):** The shape parameter **$\theta$** controls the variance of these shocks. At the macro level, $\theta$ literally becomes the aggregate elasticity of supply. A **low $\theta$** means devastating life shocks are highly variable and common, leading to a high baseline rate of pauperism regardless of economics. A **high $\theta$** means shocks are rare, and choices are dictated purely by the objective economic fundamentals ($w_j$ and $B_j$). 

### A Note on Model Limitations: The Nested Frechet
For didactic parsimony, this baseline notebook places all $2N$ options (Employment and Almshouses) into a single denominator governed by a single $\theta$. Mathematically, this assumes that navigating normal wage fluctuations and suffering catastrophic life shocks are drawn from the exact same statistical distribution. 

In reality, these distributions are profoundly different: moving to a neighboring county for a slightly better wage is a highly elastic, correlated decision, whereas entering the almshouse is an inelastic, catastrophic tail-event. When taking this framework to real-world empirical data, spatial economists resolve this by using a **Nested Frechet Model** (mathematically equivalent to a nested logit). By separating the choices into an "Employment Nest" (with a high $\theta_E$ allowing easy substitution between jobs) and an "Almshouse Nest" (with a lower $\theta_H$), the model can accurately capture the distinct variances and correlations of everyday labor markets versus catastrophic poverty.

## 4. Glossary of Parameters

| Parameter | Name | Economic Interpretation |
| :--- | :--- | :--- |
| **$N$** | Number of Locations | The total number of counties in the spatial network. |
| **$L_{0,i}$** | Baseline Population | The native-born population of county $i$ before any shocks or migration occur. |
| **$K_j$** | Fixed Capital / Land | The fixed productive capacity of county $j$. High $K$ drives labor demand and wages up. |
| **$\alpha$** | Diminishing Returns | The capital share of output. Higher $\alpha$ means wages crash harder when workers crowd into a county. |
| **$w_j$** | Nominal Wage | The endogenous clearing wage for employed workers in county $j$. |
| **$B_j$** | Relief Benefit | The fixed, objective net utility of the almshouse in county $j$ (value of food/shelter minus harshness/stigma). |
| **$d_{ij}$** | Distance Matrix | The physical or travel distance between county $i$ and county $j$. |
| **$\kappa$** | Spatial Friction | The "iceberg" cost of migration. High $\kappa$ makes it extremely costly to move to distant counties. |
| **$\epsilon$** | Idiosyncratic Shock | A random, individual-specific life event (like illness or luck) drawn from a Frechet distribution. This shock multiplies the baseline economic value to determine a person's final choice. |
| **$\theta$** | Frechet Elasticity | The inverse-variance of life shocks. High $\theta$ means people are purely rational and highly sensitive to wage/$B$ differences. Low $\theta$ means massive life shocks are common. |
| **$L_j$** | Employed Population | The endogenous number of people working in county $j$ in equilibrium. |
| **$H_j$** | Almshouse Population | The endogenous number of indigent people living in the almshouse in county $j$ in equilibrium. |

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def solve_almshouse_spatial(K_arr, B_arr, L0_arr, dist_matrix, kappa, theta, alpha, tol=1e-8, max_iter=2000):
    '''
    Solves the 2N-choice spatial equilibrium.
    K_arr: Capital array (drives labor demand/wages)
    B_arr: Relief Benefit array (net utility of the almshouse)
    L0_arr: Initial population born in each county
    dist_matrix: Pairwise distance between counties
    '''
    N = len(K_arr)
    mu = np.exp(kappa * dist_matrix)
    
    # Start with an initial guess: mostly employed, evenly spread
    L = np.copy(L0_arr) * 0.9
    
    for i in range(max_iter):
        L = np.maximum(L, 1e-6)
        
        # Calculate endogenous wages
        w = (1 - alpha) * (K_arr**alpha) * (L**-alpha)
        
        # Shapes for broadcasting: (origins i, destinations j)
        w_j = w.reshape(1, N)
        B_j = B_arr.reshape(1, N)
        
        # Fréchet utilities for Employment (E) and Almshouse (H)
        # Note: We omit A_j (amenities) for parsimony.
        u_ijE = (w_j / mu)**theta
        u_ijH = (B_j / mu)**theta
        
        # Total utility denominator for origin i (sum over all 2N options)
        u_i_total = np.sum(u_ijE, axis=1, keepdims=True) + np.sum(u_ijH, axis=1, keepdims=True)
        
        # Choice probabilities
        pi_ijE = u_ijE / u_i_total
        pi_ijH = u_ijH / u_i_total
        
        # Implied new labor and homeless populations
        L_new = np.sum(pi_ijE * L0_arr.reshape(N, 1), axis=0)
        H_new = np.sum(pi_ijH * L0_arr.reshape(N, 1), axis=0)
        
        # Check convergence on the Labor vector
        if np.max(np.abs(L_new - L)) < tol:
            return L_new, H_new, w
            
        # Heavy dampening for stability at high theta
        L = 0.9 * L + 0.1 * L_new
        
    print("Warning: Did not converge")
    return L, H_new, w

## Demo 1: Spatial Spillovers and the "Welfare Magnet" Effect

If a county decides to make its almshouse more generous (raising $B$), poverty doesn't just improve locallyit migrates. 

Let's start with a specific 5-county simulation where we set the following parameters:
* **Relief Benefit (County 3) = 0.75**: The center county has a significantly more generous almshouse than the periphery (where $B = 0.5$).
* **Friction ($\kappa$) = 1.50**: Migration is highly costly.
* **Theta ($\theta$) = 1.50**: Life shocks have high variance. Idiosyncratic bad luck is common, meaning a large baseline of people need relief regardless of the economic fundamentals.

Let's statically plot the spatial equilibrium for these exact parameters:

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N1 = 5
center1 = N1 // 2
indices1 = np.arange(N1)
dist_matrix1 = np.abs(indices1.reshape(N1, 1) - indices1.reshape(1, N1))
L0_arr1 = np.ones(N1) * 10
K_arr1 = np.ones(N1) * 10
B_arr1 = np.ones(N1) * 0.5
alpha = 0.33

B_arr_local = np.copy(B_arr1)
B_arr_local[center1] = 0.75  # Generous center

kappa_static = 1.5
theta_static = 1.5

L_eq, H_eq, w_eq = solve_almshouse_spatial(K_arr1, B_arr_local, L0_arr1, dist_matrix1, kappa_static, theta_static, alpha)

regions = [f'County {i+1}' for i in range(N1)]
colors_H = ['#d62728' if i == center1 else '#ff9896' for i in range(N1)]
colors_L = ['#1f77b4' if i == center1 else '#aec7e8' for i in range(N1)]
colors_W = ['#2ca02c' if i == center1 else '#98df8a' for i in range(N1)]

fig_static = make_subplots(rows=1, cols=2, subplot_titles=('Total Population (L_j + H_j)', 'Labor Market Wage (w_j)'))

fig_static.add_trace(go.Bar(x=regions, y=H_eq, name="Indigent (H)", marker_color=colors_H), row=1, col=1)
fig_static.add_trace(go.Bar(x=regions, y=L_eq, name="Employed (L)", marker_color=colors_L), row=1, col=1)
fig_static.add_trace(go.Bar(x=regions, y=w_eq, marker_color=colors_W, showlegend=False), row=1, col=2)

fig_static.update_layout(barmode='stack', height=450, margin=dict(l=20, r=20, t=40, b=20), paper_bgcolor="white", plot_bgcolor="white")
fig_static.update_yaxes(title_text="Population", range=[0, 15], row=1, col=1, gridcolor='lightgrey')
fig_static.update_yaxes(title_text="Nominal Wage", range=[0, 1.2], row=1, col=2, gridcolor='lightgrey')

fig_static.show()

### Economic Interpretation

This chart reveals a beautiful **General Equilibrium** dynamic that a partial-equilibrium model completely misses:

1. **The "Welfare Magnet" Effect:** Because the center county's almshouse is generous, destitute migrants flock there despite the high travel friction. County 3's *total* population swells relative to the periphery.
2. **Labor Supply Contraction:** Look closely at the blue "Employed" bars. Even though County 3 has the highest *total* population, it has the **smallest *employed* population**. The generous relief benefit acts as an attractive outside option (a high reservation utility), pulling local workers out of the active labor force.
3. **The Wage Premium:** The wage equation $w_j = (1 - \alpha) K_j^\alpha L_j^{-\alpha}$ is driven purely by the *employed* population ($L$), not the total population. Because County 3's generous welfare system "hollowed out" its active labor supply, the remaining workers are now highly scarce. This labor scarcity mathematically drives up the marginal product of labor, forcing employers to pay a **higher equilibrium wage** in the center just to retain their workforce!

### Interactive Examples

In the interactive widget below, try lowering the center's benefit to crack down on pauperism (the "Race to the Bottom"), or adjust the spatial friction $\kappa$ to see how geography limits or accelerates migration.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initial parameters
N1 = 5
center1 = N1 // 2
indices1 = np.arange(N1)
dist_matrix1 = np.abs(indices1.reshape(N1, 1) - indices1.reshape(1, N1))
L0_arr1 = np.ones(N1) * 10
K_arr1 = np.ones(N1) * 10
B_arr1 = np.ones(N1) * 0.5
alpha = 0.33

regions = [f'County {i+1}' for i in range(N1)]

# Highlight center county with bolder colors
colors_H = ['#d62728' if i == center1 else '#ff9896' for i in range(N1)]
colors_L = ['#1f77b4' if i == center1 else '#aec7e8' for i in range(N1)]
colors_W = ['#2ca02c' if i == center1 else '#98df8a' for i in range(N1)]

# Create FigureWidget with subplots
fig = go.FigureWidget(make_subplots(rows=1, cols=2, subplot_titles=('Total Population (L_j + H_j)', 'Labor Market Wage (w_j)')))

# Add Almshouse trace (trace 0) - Bottom of stack
fig.add_trace(go.Bar(x=regions, y=[0]*N1, name="Indigent (H)", marker_color=colors_H), row=1, col=1)

# Add Employed trace (trace 1) - Top of stack
fig.add_trace(go.Bar(x=regions, y=[0]*N1, name="Employed (L)", marker_color=colors_L), row=1, col=1)

# Add Wage trace (trace 2)
fig.add_trace(go.Bar(x=regions, y=[0]*N1, marker_color=colors_W, showlegend=False), row=1, col=2)

fig.update_layout(barmode='stack', height=450, margin=dict(l=20, r=20, t=40, b=20), paper_bgcolor="white", plot_bgcolor="white")
fig.update_yaxes(title_text="Population", range=[0, 15], row=1, col=1, gridcolor='lightgrey')
fig.update_yaxes(title_text="Nominal Wage", range=[0, 1.2], row=1, col=2, gridcolor='lightgrey')

# Create sliders
slider_B = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description='Benefit (Cnty 3)')
slider_kappa = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, description='Friction (kappa)')
slider_theta = widgets.FloatSlider(value=3.0, min=0.5, max=8.0, step=0.5, description='Theta')

# The update callback
def update_plot(change):
    B_center = slider_B.value
    kappa = slider_kappa.value
    theta = slider_theta.value
    
    B_arr_local = np.copy(B_arr1)
    B_arr_local[center1] = B_center
    
    L_eq, H_eq, w_eq = solve_almshouse_spatial(K_arr1, B_arr_local, L0_arr1, dist_matrix1, kappa, theta, alpha)
    
    # Efficiently mutate the traces in-place
    with fig.batch_update():
        fig.data[0].y = H_eq
        fig.data[1].y = L_eq
        fig.data[2].y = w_eq

# Observe changes
slider_B.observe(update_plot, names='value')
slider_kappa.observe(update_plot, names='value')
slider_theta.observe(update_plot, names='value')

# Initialize plot
update_plot(None)

# Display UI: Notice we display the `fig` widget directly, NOT `plt.show()`
display(widgets.VBox([slider_B, slider_kappa, slider_theta, fig]))

interactive(children=(FloatSlider(value=0.5, description='Benefit (Cnty 3)', max=0.9, min=0.1, step=0.05), Flo…

### Economic Interpretation: Macroeconomic Collapse

Notice what happens when you lower the `Global K` slider significantly (simulating a severe economic depression). The green wage line will crash *below* the red $B=0.4$ dashed line. 

If $w_j < B_j$, the interpretation fundamentally flips: poverty is no longer just about idiosyncratic bad luck. The macroeconomic fundamentals have completely collapsed, and the "relief benefit" (despite its harshness and stigma) is now genuinely better than the starvation wages offered in the labor market. When that happens, you will see a massive, structural surge in the indigent population on the left as healthy workers rationally opt out of the labor market!

## Demo 2: Macroeconomic Shocks and Centrality

Now let's examine an 11-county lattice. What happens when a macroeconomic depression hits?
By lowering the global capital `Global K`, wages drop closer to the relief baseline $B$. You will see the almshouses overflow organically as the labor market collapses.

Furthermore, pay attention to the **geographic baseline** when $K$ is healthy. Because the center of the lattice has better "market access" (it is closer to more origins), it naturally accumulates more indigent migrants than the isolated periphery!

## 5. Extension: The Nested Frechet Model

A major structural limitation of this baseline model is that we place all $2N$ options (Employment and Almshouses) into a single choice set governed by a single dispersion parameter $\theta$. Mathematically, this forces the model to assume that navigating normal wage fluctuations across counties and suffering catastrophic life shocks (like crippling injuries) are drawn from the exact same statistical distribution.

In reality, the variance of these decisions is profoundly different:
*   **The Employment Nest:** Moving to a neighboring county for a slightly better wage is a highly elastic, easily substituted decision. Workers should be highly sensitive to small wage differences, implying a high elasticity parameter $\theta_E$.
*   **The Almshouse Nest:** Entering the almshouse is an inelastic, catastrophic tail-event. People do not smoothly substitute between a factory job and an almshouse; they only choose the almshouse when hit by massive, high-variance life shocks, implying a much lower elasticity parameter $\theta_H$.

To resolve this in future iterations, we can upgrade to a **Nested Frechet** spatial model (mathematically analogous to a nested logit). In this extended framework, an individual first draws a shock for the *type* of status (Employment vs. Relief) governed by an upper-level elasticity, and then draws a conditional shock for the specific *location* governed by a lower-level elasticity. 

By separating the choices into an "Employment Nest" ($\theta_E$) and an "Almshouse Nest" ($\theta_H$), the model can accurately capture the distinct, correlated variances of everyday labor markets versus the catastrophic, uncorrelated nature of extreme poverty.

In [5]:
# Initial parameters
N2 = 11
center2 = N2 // 2
indices2 = np.arange(N2)
dist_matrix2 = np.abs(indices2.reshape(N2, 1) - indices2.reshape(1, N2))
L0_arr2 = np.ones(N2) * 10
B_arr2 = np.ones(N2) * 0.4 # Constant relief benefit across the state
x_axis = np.arange(N2) - center2

fig2 = go.FigureWidget(make_subplots(rows=1, cols=2, subplot_titles=('Total Population by Distance', 'Equilibrium Wage Gradient')))

# Add Almshouse Bar trace (Trace 0)
fig2.add_trace(go.Bar(x=x_axis, y=[0]*N2, marker_color='darkred', opacity=0.8, name='Indigent (H)'), row=1, col=1)

# Add Employed Bar trace (Trace 1)
fig2.add_trace(go.Bar(x=x_axis, y=[0]*N2, marker_color='darkblue', opacity=0.6, name='Employed (L)'), row=1, col=1)

# Add Wage Scatter trace (Trace 2)
fig2.add_trace(go.Scatter(x=x_axis, y=[0]*N2, mode='lines+markers', line=dict(color='forestgreen', width=2), marker=dict(size=8), name='Wage'), row=1, col=2)

# Add Relief Benefit Baseline
fig2.add_hline(y=0.4, line_dash="dash", line_color="red", opacity=0.5, row=1, col=2, annotation_text="Relief Benefit (B=0.4)")

fig2.update_layout(barmode='stack', height=450, margin=dict(l=20, r=20, t=40, b=20), paper_bgcolor="white", plot_bgcolor="white")
fig2.update_xaxes(title_text="Distance from Center", row=1, col=1)
fig2.update_xaxes(title_text="Distance from Center", row=1, col=2)
fig2.update_yaxes(title_text="Population", range=[0, 20], row=1, col=1, gridcolor='lightgrey')
fig2.update_yaxes(title_text="Nominal Wage (w_j)", range=[0.2, 1.5], row=1, col=2, gridcolor='lightgrey')
fig2.update_xaxes(tickmode='linear', tick0=-5, dtick=1)

# Create sliders
slider_K = widgets.FloatSlider(value=10.0, min=2.0, max=20.0, step=1.0, description='Global K')
slider_kappa2 = widgets.FloatSlider(value=0.2, min=0.0, max=1.0, step=0.1, description='Friction (kappa)')
slider_theta2 = widgets.FloatSlider(value=4.0, min=1.0, max=10.0, step=0.5, description='Theta')
slider_alpha2 = widgets.FloatSlider(value=0.33, min=0.1, max=0.9, step=0.1, description='Dim. Returns')

def update_plot2(change):
    global_K = slider_K.value
    kappa = slider_kappa2.value
    theta = slider_theta2.value
    alpha = slider_alpha2.value
    
    K_arr2 = np.ones(N2) * global_K
    
    L_eq, H_eq, w_eq = solve_almshouse_spatial(K_arr2, B_arr2, L0_arr2, dist_matrix2, kappa, theta, alpha)
    
    with fig2.batch_update():
        fig2.data[0].y = H_eq
        fig2.data[1].y = L_eq
        fig2.data[2].y = w_eq

slider_K.observe(update_plot2, names='value')
slider_kappa2.observe(update_plot2, names='value')
slider_theta2.observe(update_plot2, names='value')
slider_alpha2.observe(update_plot2, names='value')

update_plot2(None)

display(widgets.VBox([slider_K, slider_kappa2, slider_theta2, slider_alpha2, fig2]))

interactive(children=(FloatSlider(value=10.0, description='Global K', max=20.0, min=2.0, step=1.0), FloatSlide…

## Conclusion: Looking to the Data

This theoretical framework is fully generalizable to real-world data. To move from this didactic notebook to empirical estimation (e.g., using Matthew Baker's panel data on NY/New England almshouses), we would make the following structural updates:

1.  **2D Spatial Graph Networks:** Instead of a simple 1D line array, `dist_matrix` becomes a matrix of real-world distances or travel times (e.g., along 19th-century rail or canal networks) between counties.
2.  **Structural Inversion:** In the data, we observe the true populations $L_j$ and $H_j$, and perhaps proxies for $w_j$. Using techniques from Quantitative Spatial Economics (like Exact Hat Algebra or the "inversion" of the labor market clearing conditions), we can mathematically back out the unobserved local amenities ($A_j$), the harshness/stigma penalties of specific almshouses ($B_j$), and accurately estimate the core elasticity parameters ($\theta$ and $\kappa$).
3.  **Counterfactuals:** Once the model is calibrated to the panel data, we could run rigorous counterfactuals. For example: *What would have happened to the geographic distribution of poverty in New England if the 1824 County Poorhouse Act had never passed?*